# Stage B / NB 06 — Conventional CXR classifiers trained end-to-end (E0g)

Protocol reference: Section 6.2 experiment **E0g**. This notebook exists to answer referee 2d
directly: *"The method should be compared with strong baselines, including recent medical
multimodal LLMs and established CXR classification models."* NB 07 supplies the multimodal
LLMs; this one supplies the established CXR classifiers.

## The three backbones

| model | initialisation | why it is here |
| --- | --- | --- |
| DenseNet-121 | TorchXRayVision CXR weights | the de facto standard CXR classifier; domain-pretrained on ~700k radiographs |
| ConvNeXt-Tiny | ImageNet-1k | a modern convolutional architecture at comparable scale |
| ViT-B/16 | ImageNet-21k/1k | a transformer control, so "transformer" and "generative VLM" are not conflated |

DenseNet-121 with TorchXRayVision initialisation is the important one. A referee who works in
this area will expect it, and an ImageNet-initialised CNN is not a credible CXR baseline. If
`torchxrayvision` is unavailable the notebook records the fallback explicitly in
`run_config.json` rather than quietly reporting an ImageNet model under a domain-pretrained
label.

## Why these are trained end-to-end

NB 05 measures what a *frozen* representation contains. This notebook measures what a
conventional supervised model achieves when it is allowed to adapt everything. Together they
bracket the non-generative baselines, so the LoRA-VLM comparison in NB 08/09 is against a fair
upper bound rather than a straw man.

## Augmentation policy (protocol Section 5)

Implemented exactly as specified, and the forbidden list is enforced in code rather than left
to discipline:

- **allowed** — rotation ±7°, translation ±5%, scale 0.95–1.05, brightness/contrast ±10%, random 0–2% border crop
- **forbidden** — horizontal flip (laterality is part of the mRALE label), vertical flip, elastic deformation, mixup/cutmix on mRALE targets, colour jitter beyond grayscale intensity

The horizontal-flip prohibition is not pedantry: `extent_right` and `extent_left` are separate
supervised targets, so a mirrored image with unmirrored labels teaches the model that
laterality is noise.

## Outputs (under `stage_B/nb06_conventional/`)
- `predictions_conventional.jsonl`, `external_predictions.jsonl`
- `per_fold_metrics.json`, `cross_validation_aggregate_95ci_<model>.csv`, `arm_summary.csv`
- `training_curves/<model>_fold<k>.csv`, `checkpoints/<model>_fold<k>.pt`
- `run_config.json`, `gate_nb06.json`

## Gate
- Out-of-fold coverage exactly 100% per model, each image predicted once.
- AUROC computable in every fold.
- No forbidden augmentation is active.

## 1. Imports, seeds, and the Stage A path contract

In [ ]:
import gc
import json
import math
import os
import random
import sys
import time
from collections import Counter, OrderedDict, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# Shared metric definitions. Table 2 is only a valid comparison if every arm uses these.
_METRICS_SEARCH = [Path.cwd(), Path.cwd().parent, Path.cwd() / "stage_B",
                   Path.cwd().parent / "stage_B"]
for _candidate in _METRICS_SEARCH:
    if (_candidate / "cxr_metrics.py").is_file():
        sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError(
        "cxr_metrics.py not found. It must sit beside the Stage B notebooks; every arm in "
        f"Table 2 depends on its metric definitions. Searched: {_METRICS_SEARCH}")
import cxr_metrics as cm

SEED = 42
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# ---- Stage A path contract -------------------------------------------------------------
FALLBACK_STAGE_A_DIR = Path("/data/liangz2/openi/midrc/tetci_resubmit/stage_A")
PATHS_JSON_CANDIDATES = [
    FALLBACK_STAGE_A_DIR / "nb00_environment" / "stage_a_paths.json",
    Path.cwd() / "stage_a_paths.json",
    Path.cwd().parent / "stage_A" / "nb00_environment" / "stage_a_paths.json",
]
stage_paths = None
for candidate in PATHS_JSON_CANDIDATES:
    if candidate.is_file():
        stage_paths = json.loads(candidate.read_text(encoding="utf-8"))
        print("Path contract:", candidate)
        break
if stage_paths is None:
    raise FileNotFoundError("stage_a_paths.json not found. Run Stage A NB 00 first.")

PROJECT_ROOT = Path(stage_paths["project_root"])
STAGE_ROOT = Path(stage_paths["stage_root"])
STAGE_A_DIR = Path(stage_paths["stage_a_dir"])
STAGE_B_DIR = STAGE_ROOT / "stage_B"
NB01_DIR = Path(stage_paths["nb_output_dirs"]["nb01_inventory"])
NB02_DIR = Path(stage_paths["nb_output_dirs"]["nb02_folds"])
NB03_DIR = Path(stage_paths["nb_output_dirs"]["nb03_external"])
NB04_DIR = Path(stage_paths["nb_output_dirs"]["nb04_localization"])
FOLD_DEF_DIR = NB02_DIR / "fold_definitions"
MODEL_REVISIONS = stage_paths.get("model_revisions", {})

N_FOLDS = 5
INTERNAL_VALIDATION_FRACTION = 0.10   # matches the tested LoRA notebooks

print("Stage B output root:", STAGE_B_DIR)
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| BF16:", torch.cuda.is_bf16_supported())

## 2. Configuration

In [ ]:
NB06_DIR = STAGE_B_DIR / "nb06_conventional"
CHECKPOINT_DIR = NB06_DIR / "checkpoints"
CURVE_DIR = NB06_DIR / "training_curves"
for directory in [NB06_DIR, CHECKPOINT_DIR, CURVE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 224
NUM_WORKERS = 4

# Biowulf runs kernel 4.18, below accelerate's documented 5.5 minimum. The failure mode is
# DataLoader WORKER deadlock -- the classic way a ~10 GPU-hour job dies overnight with no
# traceback. persistent_workers=True makes it worse by keeping the workers alive across
# epochs. Single-process loading is slower per step but cannot deadlock.
import platform as _platform


def _kernel_tuple():
    try:
        return tuple(int(x) for x in _platform.release().split("-")[0].split(".")[:2])
    except Exception:
        return (99, 99)


OLD_KERNEL = _kernel_tuple() < (5, 5)
OVERRIDE_OLD_KERNEL_GUARD = False
if OLD_KERNEL and not OVERRIDE_OLD_KERNEL_GUARD:
    print(f"Kernel {_platform.release()} < 5.5 -> forcing NUM_WORKERS=0, pin_memory=False, "
          "persistent_workers=False to avoid worker deadlock.")
    print("  Set OVERRIDE_OLD_KERNEL_GUARD = True to keep the configured value.")
    NUM_WORKERS = 0

MODEL_SPECS = OrderedDict([
    ("densenet121_xrv", {
        "arch": "densenet121",
        "init": "torchxrayvision",
        "note": "Domain-pretrained on ~700k CXRs. The baseline a CXR referee expects.",
    }),
    ("convnext_tiny", {
        "arch": "convnext_tiny",
        "init": "imagenet",
        "note": "Modern convolutional control at comparable scale.",
    }),
    ("vit_b16", {
        "arch": "vit_b_16",
        "init": "imagenet",
        "note": "Transformer control, so architecture family is not confounded with 'VLM'.",
    }),
])

RUN_MODELS = list(MODEL_SPECS)
EPOCHS = 30
BATCH_SIZE = 32
LEARNING_RATE = 3e-4
BACKBONE_LEARNING_RATE = 3e-5      # lower LR for pretrained weights than for the fresh heads
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 2
PATIENCE = 6
GRAD_CLIP_NORM = 1.0
USE_AMP = True

# Augmentation, protocol Section 5. FORBIDDEN_AUGMENTATIONS is asserted in the gate.
AUGMENTATION = {
    "rotation_degrees": 7.0,
    "translate_fraction": 0.05,
    "scale_range": (0.95, 1.05),
    "brightness_contrast_jitter": 0.10,
    "border_crop_fraction": 0.02,
}
FORBIDDEN_AUGMENTATIONS = {
    "horizontal_flip": False,      # laterality is a supervised target
    "vertical_flip": False,
    "elastic_deformation": False,
    "mixup": False,
    "cutmix": False,
    "colour_jitter": False,
}

# E8 hooks; off by default so E0g is a clean baseline.
USE_CLASS_BALANCED_SAMPLER = False
USE_POSITIVE_WEIGHT = False

RUN_EXTERNAL = True
# Resume control. A fold is ~40 minutes and the whole notebook is ~10 GPU-hours, so both
# levels of resume are on by default: completed folds are skipped, and an interrupted fold
# restarts from its last completed epoch rather than from scratch.
RESUME_MID_FOLD = True
FORCE_RETRAIN = False        # True ignores every saved fold and retrains from scratch

SMOKE_TEST = False                 # True -> 2 epochs, fold 0 only, tiny subset

if SMOKE_TEST:
    EPOCHS, RUN_MODELS = 2, RUN_MODELS[:1]

print("Models:", RUN_MODELS)
print("Epochs:", EPOCHS, "| batch:", BATCH_SIZE, "| image size:", IMAGE_SIZE)
print("Output:", NB06_DIR)

## 3. Load the cohort table

In [ ]:
def load_view_index():
    path = NB04_DIR / "view_index.csv"
    if not path.is_file():
        raise FileNotFoundError(f"{path} not found. Run Stage A NB 04 first.")
    frame = pd.read_csv(path)
    print(f"view_index.csv: {len(frame):,} rows, cohorts={dict(Counter(frame['cohort']))}")
    return frame


def load_folds():
    path = FOLD_DEF_DIR / "midrc_folds_v2.csv"
    if not path.is_file():
        raise FileNotFoundError(
            f"{path} not found. Run Stage A NB 02 first. Do NOT fall back to the legacy "
            "multi_task_CV folds: they leak at study level."
        )
    frame = pd.read_csv(path)
    print(f"midrc_folds_v2.csv: {len(frame):,} images, "
          f"{frame['group_id'].nunique():,} groups, folds={dict(sorted(Counter(frame['fold']).items()))}")
    return frame


def build_cohort_table():
    # One row per image: labels + fold + every cached view path. This is the single table
    # every Stage B notebook trains and predicts from.
    views = load_view_index()
    folds = load_folds()

    internal = folds.merge(
        views[views["cohort"] == "MIDRC"].drop(columns=["held_out_fold"], errors="ignore"),
        on="filename", how="inner", suffixes=("", "_view"),
    )
    if len(internal) != len(folds):
        missing = set(folds["filename"]) - set(internal["filename"])
        raise RuntimeError(
            f"{len(missing)} fold images have no NB 04 localization row (e.g. "
            f"{sorted(missing)[:5]}). Re-run NB 04 with MAX_IMAGES=None."
        )
    internal["mrale_right"] = (internal["extent_right_numerical"]
                               * internal["density_right_numerical"])
    internal["mrale_left"] = (internal["extent_left_numerical"]
                              * internal["density_left_numerical"])
    internal["is_external"] = False

    external_rows = []
    external_dir = NB03_DIR / "external_manifests"
    if external_dir.is_dir():
        for manifest_path in sorted(external_dir.glob("*_manifest.csv")):
            frame = pd.read_csv(manifest_path)
            if "status" in frame.columns:
                frame = frame[frame["status"] == "OK"]
            if not len(frame):
                continue
            cohort = str(frame["cohort"].iloc[0])
            merged = frame.merge(
                views[views["cohort"] == cohort][
                    ["filename", "v0_image", "v1_thorax_image", "v2_left_image",
                     "v2_right_image", "left_box", "right_box", "any_fallback"]
                ],
                on="filename", how="inner",
            )
            merged["fold"] = -1
            merged["is_external"] = True
            merged["group_id"] = "external::" + merged["filename"].astype(str)
            for column in ["mrale_total_annotated", "mrale_right", "mrale_left",
                           "extent_right_numerical", "density_right_numerical",
                           "extent_left_numerical", "density_left_numerical"]:
                if column not in merged.columns:
                    merged[column] = np.nan
            if "mrale_total" in merged.columns:
                merged["mrale_total_annotated"] = merged["mrale_total"]
            external_rows.append(merged)

    table = pd.concat([internal] + external_rows, ignore_index=True, sort=False)
    table["image_key"] = table.apply(
        lambda row: f"{row.get('cohort', 'MIDRC')}::{row['filename']}", axis=1)
    print()
    print(f"Cohort table: {len(table):,} rows "
          f"({int((~table['is_external']).sum()):,} internal, "
          f"{int(table['is_external'].sum()):,} external)")
    return table


def grouped_inner_split(subset, fraction, seed):
    # Group-aware inner validation split, same construction as the tested notebooks: whole
    # groups move together so the inner split cannot leak either.
    groups = sorted(subset["group_id"].astype(str).unique())
    rng = random.Random(seed)
    rng.shuffle(groups)
    n_validation = max(1, round(len(groups) * fraction))
    validation_groups = set(groups[:n_validation])
    is_validation = subset["group_id"].astype(str).isin(validation_groups)
    train, validation = subset[~is_validation], subset[is_validation]
    assert not (set(train["group_id"]) & set(validation["group_id"]))
    return train, validation


def ground_truth_fields(row):
    def maybe_int(value):
        return None if value is None or (isinstance(value, float) and math.isnan(value)) else int(value)
    covid = row.get("covid_positive")
    if isinstance(covid, float) and math.isnan(covid):
        covid = None
    return {
        "gt_covid": covid if covid in {"Yes", "No"} else None,
        "gt_mrale_total": maybe_int(row.get("mrale_total_annotated")),
        "gt_mrale_right": maybe_int(row.get("mrale_right")),
        "gt_mrale_left": maybe_int(row.get("mrale_left")),
        "gt_extent_right": maybe_int(row.get("extent_right_numerical")),
        "gt_density_right": maybe_int(row.get("density_right_numerical")),
        "gt_extent_left": maybe_int(row.get("extent_left_numerical")),
        "gt_density_left": maybe_int(row.get("density_left_numerical")),
    }


def evaluate_arm(rows, label):
    # Single entry point for metrics, so every arm in Table 2 is scored identically.
    covid_rows = [row for row in rows if row.get("gt_covid") is not None]
    metrics = {"arm": label, "n_rows": len(rows)}
    if covid_rows:
        metrics["covid"] = cm.classification_metrics(
            [row["gt_covid"] for row in covid_rows],
            [row.get("covid_pred") for row in covid_rows],
            [row.get("covid_score") for row in covid_rows],
        )
    mrale_rows = [row for row in rows if row.get("gt_mrale_total") is not None]
    if mrale_rows:
        metrics["mrale"] = cm.mrale_metrics(mrale_rows)
    metrics["output"] = cm.localization_free_metrics(rows)
    return metrics


def print_arm_summary(metrics):
    covid = metrics.get("covid", {})
    mrale = metrics.get("mrale", {})
    print(f"  {metrics['arm']:<34} "
          f"AUROC={covid.get('auroc', float('nan')):.4f} "
          f"balAcc={covid.get('balanced_accuracy', float('nan')):.4f} "
          f"spec={covid.get('specificity', float('nan')):.4f} | "
          f"mRALE MAE={mrale.get('mae', float('nan')):.3f} "
          f"QWK={mrale.get('qwk', float('nan')):.4f} "
          f"cov={mrale.get('coverage', float('nan')):.3f}")

In [ ]:
cohort = build_cohort_table()
internal_cohort = cohort[~cohort["is_external"]].reset_index(drop=True)
external_cohort = cohort[cohort["is_external"]].reset_index(drop=True)
if SMOKE_TEST:
    internal_cohort = internal_cohort.groupby("fold", group_keys=False).head(40)
    external_cohort = external_cohort.head(8)
    print(f"SMOKE TEST: {len(internal_cohort)} internal, {len(external_cohort)} external")
print()
print("PCR balance:", dict(internal_cohort["covid_positive"].value_counts()))

## 4. Dataset and augmentation

Augmentation is written out explicitly with torchvision transforms rather than assembled from
a preset, so that the forbidden list is verifiable by reading the code. `build_transform`
returns the transform *and* a manifest of what it applied, and the manifest is what the gate
checks.

In [ ]:
from PIL import Image
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms

Image.MAX_IMAGE_PIXELS = None

# CXR-appropriate normalisation: single-channel statistics replicated across 3 channels, since
# every backbone here expects 3-channel input.
CXR_MEAN, CXR_STD = [0.498, 0.498, 0.498], [0.252, 0.252, 0.252]


def build_transform(training):
    steps, manifest = [], {}
    if training:
        steps.append(transforms.RandomAffine(
            degrees=AUGMENTATION["rotation_degrees"],
            translate=(AUGMENTATION["translate_fraction"],) * 2,
            scale=AUGMENTATION["scale_range"],
            interpolation=transforms.InterpolationMode.BILINEAR,
            fill=0,
        ))
        manifest["random_affine"] = {
            "degrees": AUGMENTATION["rotation_degrees"],
            "translate": AUGMENTATION["translate_fraction"],
            "scale": list(AUGMENTATION["scale_range"]),
        }
        jitter = AUGMENTATION["brightness_contrast_jitter"]
        steps.append(transforms.ColorJitter(brightness=jitter, contrast=jitter,
                                            saturation=0.0, hue=0.0))
        manifest["brightness_contrast_jitter"] = jitter
        manifest["saturation_hue_jitter"] = 0.0    # grayscale intensity only
        crop = AUGMENTATION["border_crop_fraction"]
        steps.append(transforms.RandomResizedCrop(
            IMAGE_SIZE, scale=(1.0 - 2 * crop, 1.0), ratio=(0.98, 1.02),
            interpolation=transforms.InterpolationMode.BILINEAR,
        ))
        manifest["random_border_crop_fraction"] = crop
    else:
        steps.append(transforms.Resize(
            (IMAGE_SIZE, IMAGE_SIZE),
            interpolation=transforms.InterpolationMode.BILINEAR))
        manifest["resize_only"] = IMAGE_SIZE
    steps.extend([transforms.ToTensor(), transforms.Normalize(CXR_MEAN, CXR_STD)])
    manifest["normalize"] = {"mean": CXR_MEAN, "std": CXR_STD}
    # Recorded explicitly so the gate can assert none of these were switched on.
    manifest["forbidden_applied"] = {name: False for name in FORBIDDEN_AUGMENTATIONS}
    return transforms.Compose(steps), manifest


_, TRAIN_AUGMENTATION_MANIFEST = build_transform(training=True)
_, EVAL_AUGMENTATION_MANIFEST = build_transform(training=False)
print("Train augmentation manifest:")
print(json.dumps(TRAIN_AUGMENTATION_MANIFEST, indent=2))


class CxrDataset(Dataset):
    def __init__(self, frame, training):
        # to_dict("records") ONCE instead of frame.iloc[i] per sample. On a wide mixed-dtype
        # frame pandas rebuilds an interleaved dtype on every .iloc row access, which is the
        # cost that made the equivalent loop in NB 05 appear to hang.
        self.records = frame.reset_index(drop=True).to_dict("records")
        self.transform, _ = build_transform(training)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        with Image.open(row["v0_image"]) as handle:
            image = handle.convert("RGB")
        tensor = self.transform(image)

        def label_or(value, default=-1):
            return default if value is None or pd.isna(value) else int(value)

        covid = row.get("covid_positive")
        return {
            "image": tensor,
            "index": index,
            "covid": torch.tensor(
                1.0 if covid == "Yes" else 0.0, dtype=torch.float32),
            "covid_valid": torch.tensor(covid in {"Yes", "No"}, dtype=torch.bool),
            "extent_right": torch.tensor(label_or(row.get("extent_right_numerical")), dtype=torch.long),
            "density_right": torch.tensor(label_or(row.get("density_right_numerical")), dtype=torch.long),
            "extent_left": torch.tensor(label_or(row.get("extent_left_numerical")), dtype=torch.long),
            "density_left": torch.tensor(label_or(row.get("density_left_numerical")), dtype=torch.long),
        }


def make_loader(frame, training, batch_size=None):
    dataset = CxrDataset(frame, training)
    batch_size = batch_size or BATCH_SIZE
    sampler = None
    shuffle = training
    if training and USE_CLASS_BALANCED_SAMPLER:
        labels = (frame["covid_positive"] == "Yes").astype(int).to_numpy()
        counts = np.bincount(labels, minlength=2).astype(float)
        weights = np.where(labels == 1, 1.0 / max(counts[1], 1), 1.0 / max(counts[0], 1))
        sampler = WeightedRandomSampler(
            torch.as_tensor(weights, dtype=torch.double), num_samples=len(frame),
            replacement=True, generator=torch.Generator().manual_seed(SEED))
        shuffle = False
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available() and not (OLD_KERNEL
                                                      and not OVERRIDE_OLD_KERNEL_GUARD),
        drop_last=False, persistent_workers=NUM_WORKERS > 0,
    )


# ---- Batch preflight ---------------------------------------------------------------------
# Cheap insurance before ~10 GPU-hours across three backbones and five folds. Confirm one real
# batch carries image tensors of the expected shape and that the labels are not all missing.
# A silent failure here trains on blank inputs and looks like a weak result, not a bug.
_probe_frame = internal_cohort.head(4) if "internal_cohort" in dir() else None
if _probe_frame is not None and len(_probe_frame):
    _probe_loader = make_loader(_probe_frame, training=False, batch_size=2)
    _batch = next(iter(_probe_loader))
    print()
    print("Batch preflight")
    print(f"  image tensor : {tuple(_batch['image'].shape)} dtype={_batch['image'].dtype}")
    print(f"  intensity    : min={_batch['image'].min():.3f} max={_batch['image'].max():.3f} "
          f"std={_batch['image'].std():.3f}")
    print(f"  covid valid  : {int(_batch['covid_valid'].sum())}/{len(_batch['covid_valid'])}")
    print(f"  extent_right : {_batch['extent_right'].tolist()}")
    if _batch["image"].std() < 1e-3:
        raise RuntimeError(
            "Batch preflight FAILED: images are effectively constant, so the model would train "
            "on blank inputs. Check that v0_image paths resolve and that the transform is not "
            "collapsing the intensity range."
        )
    if (_batch["extent_right"] < 0).all():
        raise RuntimeError(
            "Batch preflight FAILED: every mRALE label is -1 (missing). The label columns are "
            "not reaching the dataset; check the cohort merge in build_cohort_table()."
        )
    print("  OK -- images and labels both present.")


## 5. Model construction

Every backbone gets the same multi-task head so the comparison is about representation, not
head capacity: one COVID logit plus four CORAL ordinal blocks (extent 0-4, density 0-3, per
lung). mRALE total is reconstructed as `er*dr + el*dl`, so arithmetic consistency holds by
construction — the same head design as NB 05, which keeps E0d/E0e/E0f and E0g comparable.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

COMPONENTS = OrderedDict([
    ("extent_right", 5), ("density_right", 4), ("extent_left", 5), ("density_left", 4),
])


class CoralBlock(nn.Module):
    def __init__(self, in_features, n_classes):
        super().__init__()
        self.n_classes = n_classes
        self.projection = nn.Linear(in_features, 1, bias=False)
        self.thresholds = nn.Parameter(torch.zeros(n_classes - 1))

    def forward(self, features):
        return self.projection(features) + self.thresholds

    @staticmethod
    def loss(logits, targets, n_classes):
        levels = torch.arange(n_classes - 1, device=logits.device)[None, :]
        binary = (targets[:, None] > levels).float()
        return F.binary_cross_entropy_with_logits(logits, binary)

    @staticmethod
    def to_distribution(logits):
        greater = torch.cummin(torch.sigmoid(logits), dim=1).values
        ones = torch.ones_like(greater[:, :1])
        zeros = torch.zeros_like(greater[:, :1])
        return (torch.cat([ones, greater], dim=1)
                - torch.cat([greater, zeros], dim=1)).clamp_min(0.0)


class MultiTaskCxrModel(nn.Module):
    def __init__(self, backbone, feature_dim):
        super().__init__()
        self.backbone = backbone
        self.covid = nn.Linear(feature_dim, 1)
        self.blocks = nn.ModuleDict({
            component: CoralBlock(feature_dim, n_classes)
            for component, n_classes in COMPONENTS.items()
        })

    def forward(self, images):
        features = self.backbone(images)
        if features.ndim > 2:
            features = torch.flatten(features, 1)
        outputs = {"covid": self.covid(features).squeeze(-1)}
        for component, block in self.blocks.items():
            outputs[component] = block(features)
        return outputs


def build_backbone(name):
    spec = MODEL_SPECS[name]
    arch, init = spec["arch"], spec["init"]
    provenance = {"arch": arch, "requested_init": init}

    if init == "torchxrayvision":
        try:
            import torchxrayvision as xrv
            source = xrv.models.DenseNet(weights="densenet121-res224-all")
            backbone = source.features
            feature_dim = 1024
            module = nn.Sequential(
                backbone, nn.ReLU(inplace=True), nn.AdaptiveAvgPool2d(1), nn.Flatten())
            provenance.update({
                "actual_init": "torchxrayvision:densenet121-res224-all",
                "domain_pretrained": True,
            })
            return module, feature_dim, provenance
        except Exception as exc:
            provenance["torchxrayvision_error"] = f"{type(exc).__name__}: {exc}"
            provenance["actual_init"] = "imagenet_FALLBACK"
            provenance["domain_pretrained"] = False
            print(f"  WARNING: torchxrayvision unavailable ({exc}).")
            print("  Falling back to ImageNet DenseNet-121. This is recorded in "
                  "run_config.json and MUST be disclosed: an ImageNet CNN is not a "
                  "domain-pretrained CXR baseline, and the manuscript cannot label it one.")

    from torchvision import models
    if arch == "densenet121":
        source = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        module = nn.Sequential(
            source.features, nn.ReLU(inplace=True), nn.AdaptiveAvgPool2d(1), nn.Flatten())
        feature_dim = 1024
    elif arch == "convnext_tiny":
        source = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        feature_dim = 768
        source.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.LayerNorm(feature_dim))
        module = source
    elif arch == "vit_b_16":
        source = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
        feature_dim = source.hidden_dim
        source.heads = nn.Identity()
        module = source
    else:
        raise ValueError(f"Unknown architecture {arch}")
    provenance.setdefault("actual_init", "imagenet")
    provenance.setdefault("domain_pretrained", False)
    return module, feature_dim, provenance


for name in RUN_MODELS:
    print(f"{name}: {MODEL_SPECS[name]['note']}")

## 6. Train / select / predict, per fold

Selection is on inner validation with the same composite criterion as NB 05
(`MAE + 8*(1 - AUROC)`), so the two notebooks pick checkpoints on identical grounds. Parameter
groups give the pretrained backbone a lower learning rate than the fresh heads, which is
standard for transfer learning and avoids destroying domain pretraining in the first epochs —
the exact risk for the TorchXRayVision DenseNet.

In [ ]:
def decode_outputs(outputs):
    distributions = {
        component: CoralBlock.to_distribution(outputs[component])
        for component in COMPONENTS
    }
    hard, expected, variance = {}, {}, {}
    for component, distribution in distributions.items():
        values = torch.arange(distribution.shape[1], dtype=torch.float32,
                              device=distribution.device)
        normalised = distribution / distribution.sum(dim=1, keepdim=True).clamp_min(1e-9)
        hard[component] = normalised.argmax(dim=1)
        mean = (normalised * values).sum(dim=1)
        expected[component] = mean
        variance[component] = (normalised * (values - mean[:, None]) ** 2).sum(dim=1)
    right = hard["extent_right"] * hard["density_right"]
    left = hard["extent_left"] * hard["density_left"]
    return {
        "hard": hard, "mrale_right": right, "mrale_left": left, "mrale_total": right + left,
        "mrale_total_expected": (expected["extent_right"] * expected["density_right"]
                                 + expected["extent_left"] * expected["density_left"]),
        "uncertainty": sum(variance.values()) / len(variance),
        "covid_score": torch.sigmoid(outputs["covid"]),
    }


@torch.no_grad()
def run_inference(model, loader, frame, device):
    model.eval()
    collected = defaultdict(list)
    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.bfloat16,
                            enabled=USE_AMP and torch.cuda.is_available()):
            outputs = model(images)
        outputs = {key: value.float() for key, value in outputs.items()}
        decoded = decode_outputs(outputs)
        collected["index"].append(batch["index"])
        collected["covid_score"].append(decoded["covid_score"].cpu())
        collected["mrale_total"].append(decoded["mrale_total"].cpu())
        collected["mrale_right"].append(decoded["mrale_right"].cpu())
        collected["mrale_left"].append(decoded["mrale_left"].cpu())
        collected["mrale_total_expected"].append(decoded["mrale_total_expected"].cpu())
        collected["uncertainty"].append(decoded["uncertainty"].cpu())
        for component in COMPONENTS:
            collected[component].append(decoded["hard"][component].cpu())
    return {key: torch.cat(value) for key, value in collected.items()}


_RECORD_CACHE = {}


def frame_records(frame):
    """Cache to_dict("records") per frame object; called once per epoch otherwise."""
    key = id(frame)
    if key not in _RECORD_CACHE:
        _RECORD_CACHE[key] = frame.reset_index(drop=True).to_dict("records")
    return _RECORD_CACHE[key]


def rows_from_inference(collected, frame, model_name, fold, positive_weight_used):
    rows = []
    order = collected["index"].tolist()
    records = frame_records(frame)
    for position, frame_index in enumerate(order):
        source = records[int(frame_index)]
        score = float(collected["covid_score"][position])
        rows.append(cm.make_prediction_row(
            image_key=str(source["image_key"]), cohort=source.get("cohort", "MIDRC"),
            subcohort=source.get("subcohort", "MIDRC"), filename=source["filename"],
            held_out_fold=fold, agent=f"E0g_{model_name}", arm=f"E0g_{model_name}",
            view="v0", task="joint",
            covid_pred="Yes" if score >= 0.5 else "No", covid_score=score,
            mrale_total=int(collected["mrale_total"][position]),
            mrale_right=int(collected["mrale_right"][position]),
            mrale_left=int(collected["mrale_left"][position]),
            extent_right=int(collected["extent_right"][position]),
            density_right=int(collected["density_right"][position]),
            extent_left=int(collected["extent_left"][position]),
            density_left=int(collected["density_left"][position]),
            mrale_total_expected=float(collected["mrale_total_expected"][position]),
            mrale_uncertainty=float(collected["uncertainty"][position]),
            valid=True, parse_error=None,
            model_id=model_name, model_revision=None,
            positive_weight_used=positive_weight_used,
            **ground_truth_fields(source),
        ))
    return rows


def composite_selection_score(rows):
    mrale = cm.mrale_metrics([row for row in rows if row["gt_mrale_total"] is not None])
    covid = cm.classification_metrics(
        [row["gt_covid"] for row in rows], [row["covid_pred"] for row in rows],
        [row["covid_score"] for row in rows])
    mae = mrale.get("mae", float("inf"))
    auroc = covid.get("auroc", float("nan"))
    penalty = 0.0 if math.isnan(auroc) else (1.0 - auroc) * 8.0
    return mae + penalty, mae, auroc

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
predictions = []
per_fold_metrics = defaultdict(dict)
model_provenance = {}
fold_diagnostics = {}
checkpoint_paths = {}

for model_name in RUN_MODELS:
    print("=" * 78)
    print("MODEL:", model_name)
    model_rows = []
    for fold in range(N_FOLDS):
        fold_dir = NB06_DIR / "folds" / f"{model_name}_fold{fold}"
        fold_dir.mkdir(parents=True, exist_ok=True)
        fold_summary_path = fold_dir / "fold_summary.json"
        fold_predictions_path = fold_dir / "predictions.jsonl"
        resume_path = fold_dir / "resume.pt"

        # ---- Skip a fold already finished under an identical configuration ---------------
        # Ten GPU-hours across three models and five folds each; without this, any
        # interruption restarts every completed fold.
        fold_config = {"epochs": EPOCHS, "batch_size": BATCH_SIZE, "image_size": IMAGE_SIZE,
                       "head_lr": LEARNING_RATE, "backbone_lr": BACKBONE_LEARNING_RATE,
                       "augmentation": AUGMENTATION,
                       "class_balanced_sampler": USE_CLASS_BALANCED_SAMPLER,
                       "positive_weight": USE_POSITIVE_WEIGHT}
        if (not FORCE_RETRAIN and fold_summary_path.is_file()
                and fold_predictions_path.is_file()):
            try:
                saved = json.loads(fold_summary_path.read_text(encoding="utf-8"))
            except Exception:
                saved = None
            if saved and saved.get("fold_config") == fold_config:
                print(f"  fold {fold}: already complete under this config; reusing "
                      f"{fold_predictions_path.name}")
                saved_rows = cm.read_jsonl(fold_predictions_path)
                model_rows.extend(saved_rows)
                per_fold_metrics[model_name][fold] = saved["generation_metrics"]
                fold_diagnostics[f"{model_name}/fold{fold}"] = saved["diagnostics"]
                model_provenance.setdefault(model_name, saved.get("provenance", {}))
                checkpoint_paths[f"{model_name}_fold{fold}"] = saved.get("checkpoint")
                print_arm_summary(saved["generation_metrics"])
                continue
            print(f"  fold {fold}: saved summary used a DIFFERENT config; retraining.")

        test_subset = internal_cohort[internal_cohort["fold"] == fold]
        pool = internal_cohort[internal_cohort["fold"] != fold]
        train_subset, validation_subset = grouped_inner_split(
            pool, INTERNAL_VALIDATION_FRACTION, SEED + fold)

        torch.manual_seed(SEED + fold)
        backbone, feature_dim, provenance = build_backbone(model_name)
        model_provenance[model_name] = provenance
        model = MultiTaskCxrModel(backbone, feature_dim).to(device)
        n_parameters = sum(p.numel() for p in model.parameters())
        n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

        head_parameters = [p for name, p in model.named_parameters()
                           if not name.startswith("backbone")]
        backbone_parameters = [p for name, p in model.named_parameters()
                               if name.startswith("backbone")]
        optimizer = torch.optim.AdamW([
            {"params": backbone_parameters, "lr": BACKBONE_LEARNING_RATE},
            {"params": head_parameters, "lr": LEARNING_RATE},
        ], weight_decay=WEIGHT_DECAY)

        train_loader = make_loader(train_subset, training=True)
        validation_loader = make_loader(validation_subset, training=False)
        test_loader = make_loader(test_subset, training=False)

        steps_per_epoch = max(1, len(train_loader))
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=[BACKBONE_LEARNING_RATE, LEARNING_RATE],
            total_steps=EPOCHS * steps_per_epoch,
            pct_start=min(0.3, WARMUP_EPOCHS / max(EPOCHS, 1)), anneal_strategy="cos",
        )

        pos_weight = None
        if USE_POSITIVE_WEIGHT:
            positives = int((train_subset["covid_positive"] == "Yes").sum())
            negatives = int((train_subset["covid_positive"] == "No").sum())
            if positives and negatives:
                pos_weight = torch.tensor(negatives / positives, dtype=torch.float32,
                                          device=device)

        best_score, best_state, best_epoch, stale = float("inf"), None, -1, 0
        history = []
        start_epoch = 0

        # ---- Mid-fold resume -------------------------------------------------------------
        # A single fold is ~40 minutes. resume.pt is rewritten after every epoch, so an
        # interruption costs at most one epoch rather than the whole fold.
        if RESUME_MID_FOLD and resume_path.is_file():
            try:
                state = torch.load(resume_path, map_location="cpu")
                if state.get("fold_config") == fold_config:
                    model.load_state_dict(state["model"])
                    optimizer.load_state_dict(state["optimizer"])
                    scheduler.load_state_dict(state["scheduler"])
                    start_epoch = int(state["epoch"]) + 1
                    best_score = float(state["best_score"])
                    best_epoch = int(state["best_epoch"])
                    stale = int(state["stale"])
                    history = list(state.get("history", []))
                    best_state = state.get("best_state")
                    print(f"    resuming fold {fold} at epoch {start_epoch} "
                          f"(best so far {best_score:.3f} @ epoch {best_epoch})")
                else:
                    print("    resume.pt was written under a different config; ignoring it.")
            except Exception as exc:
                print(f"    could not read resume.pt ({type(exc).__name__}: {exc}); "
                      "starting this fold from scratch.")

        started = time.perf_counter()
        for epoch in range(start_epoch, EPOCHS):
            model.train()
            epoch_loss, n_seen = 0.0, 0
            for batch in train_loader:
                images = batch["image"].to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast("cuda", dtype=torch.bfloat16,
                                    enabled=USE_AMP and torch.cuda.is_available()):
                    outputs = model(images)
                loss = torch.zeros((), device=device)
                covid_valid = batch["covid_valid"].to(device)
                if covid_valid.any():
                    loss = loss + F.binary_cross_entropy_with_logits(
                        outputs["covid"].float()[covid_valid],
                        batch["covid"].to(device)[covid_valid],
                        pos_weight=pos_weight)
                for component, n_classes in COMPONENTS.items():
                    targets = batch[component].to(device)
                    mask = targets >= 0
                    if mask.any():
                        loss = loss + CoralBlock.loss(
                            outputs[component].float()[mask], targets[mask], n_classes)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                optimizer.step()
                scheduler.step()
                epoch_loss += float(loss) * images.size(0)
                n_seen += images.size(0)

            validation_rows = rows_from_inference(
                run_inference(model, validation_loader, validation_subset, device),
                validation_subset, model_name, fold, pos_weight is not None)
            score, mae, auroc = composite_selection_score(validation_rows)
            history.append({"epoch": epoch, "train_loss": epoch_loss / max(n_seen, 1),
                            "val_composite": score, "val_mae": mae, "val_auroc": auroc,
                            "lr_backbone": optimizer.param_groups[0]["lr"],
                            "lr_head": optimizer.param_groups[1]["lr"]})
            if score < best_score - 1e-6:
                best_score, best_epoch, stale = score, epoch, 0
                best_state = {key: value.detach().cpu().clone()
                              for key, value in model.state_dict().items()}
            else:
                stale += 1
            print(f"    epoch {epoch:02d} loss={epoch_loss / max(n_seen, 1):.4f} "
                  f"val_mae={mae:.3f} val_auroc={auroc:.4f} composite={score:.3f}"
                  + ("  <- best" if best_epoch == epoch else ""))
            # Durable after every epoch. Saving best_state too means the selected checkpoint
            # survives an interruption, not just the training position.
            torch.save({
                "fold_config": fold_config, "epoch": epoch,
                "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "best_score": best_score,
                "best_epoch": best_epoch, "stale": stale, "history": history,
                "best_state": best_state,
            }, resume_path)

            if stale >= PATIENCE:
                print(f"    early stop at epoch {epoch} (patience {PATIENCE})")
                break

        if best_state is not None:
            model.load_state_dict(best_state)
        checkpoint_path = CHECKPOINT_DIR / f"{model_name}_fold{fold}.pt"
        torch.save({"state_dict": best_state, "best_epoch": best_epoch,
                    "feature_dim": feature_dim, "provenance": provenance}, checkpoint_path)
        checkpoint_paths[f"{model_name}_fold{fold}"] = str(checkpoint_path)

        elapsed = time.perf_counter() - started
        fold_rows = rows_from_inference(
            run_inference(model, test_loader, test_subset, device),
            test_subset, model_name, fold, pos_weight is not None)
        for row in fold_rows:
            row["seconds"] = round(elapsed / max(len(fold_rows), 1), 6)
        model_rows.extend(fold_rows)

        metrics = evaluate_arm(fold_rows, f"E0g_{model_name}/fold{fold}")
        per_fold_metrics[model_name][fold] = metrics
        fold_diagnostics[f"{model_name}/fold{fold}"] = {
            "best_epoch": best_epoch, "epochs_run": len(history),
            "best_val_composite": best_score, "train_seconds": round(elapsed, 1),
            "total_parameters": int(n_parameters), "trainable_parameters": int(n_trainable),
            "peak_memory_gib": (round(torch.cuda.max_memory_allocated() / 1024 ** 3, 3)
                                if torch.cuda.is_available() else None),
        }
        pd.DataFrame(history).to_csv(CURVE_DIR / f"{model_name}_fold{fold}.csv", index=False)

        cm.write_jsonl(fold_predictions_path, fold_rows)
        cm.write_json(fold_summary_path, {
            "model": model_name, "fold": fold, "fold_config": fold_config,
            "provenance": provenance,
            "diagnostics": fold_diagnostics[f"{model_name}/fold{fold}"],
            "generation_metrics": metrics,
            "checkpoint": str(checkpoint_path),
        })
        # The fold is complete, so the mid-fold resume state is no longer needed.
        resume_path.unlink(missing_ok=True)

        print_arm_summary(metrics)

        del model, backbone, optimizer, scheduler, train_loader, validation_loader, test_loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

    predictions.extend(model_rows)
    print()
    print("  POOLED out-of-fold:")
    print_arm_summary(evaluate_arm(model_rows, f"E0g_{model_name}/pooled"))

cm.write_jsonl(NB06_DIR / "predictions_conventional.jsonl", predictions)
print()
print(f"Wrote {len(predictions):,} predictions")

## 7. Metrics and arm summary

In [ ]:
summary_rows = []
for model_name in RUN_MODELS:
    model_rows = [row for row in predictions if row["arm"] == f"E0g_{model_name}"]
    pooled = evaluate_arm(model_rows, f"E0g_{model_name}")
    aggregate = cm.aggregate_over_folds(per_fold_metrics[model_name])
    pd.DataFrame(aggregate).to_csv(
        NB06_DIR / f"cross_validation_aggregate_95ci_{model_name}.csv", index=False)

    def fold_ci(metric):
        match = [row for row in aggregate if row["metric"] == metric]
        return (match[0]["mean"], match[0]["ci95_lower"], match[0]["ci95_upper"]) if match else (None, None, None)

    covid, mrale = pooled.get("covid", {}), pooled.get("mrale", {})
    auroc_mean, auroc_low, auroc_high = fold_ci("covid.auroc")
    mae_mean, mae_low, mae_high = fold_ci("mrale.mae")
    diagnostics = fold_diagnostics.get(f"{model_name}/fold0", {})
    summary_rows.append(OrderedDict([
        ("arm", f"E0g_{model_name}"),
        ("init", model_provenance.get(model_name, {}).get("actual_init")),
        ("domain_pretrained", model_provenance.get(model_name, {}).get("domain_pretrained")),
        ("total_parameters_M", round(diagnostics.get("total_parameters", 0) / 1e6, 1)),
        ("n_images", pooled["n_rows"]),
        ("mrale_mae_pooled", round(mrale.get("mae", float("nan")), 3)),
        ("mrale_mae_ci95", None if mae_mean is None else f"[{mae_low:.3f}, {mae_high:.3f}]"),
        ("covid_auroc_pooled", round(covid.get("auroc", float("nan")), 4)),
        ("covid_auroc_ci95", None if auroc_mean is None else f"[{auroc_low:.4f}, {auroc_high:.4f}]"),
        ("covid_auprc", round(covid.get("auprc", float("nan")), 4)),
        ("covid_balanced_accuracy", round(covid.get("balanced_accuracy", float("nan")), 4)),
        ("covid_sensitivity", round(covid.get("sensitivity", float("nan")), 4)),
        ("covid_specificity", round(covid.get("specificity", float("nan")), 4)),
        ("covid_f1", round(covid.get("f1", float("nan")), 4)),
        ("covid_mcc", round(covid.get("mcc", float("nan")), 4)),
        ("covid_brier", round(covid.get("brier", float("nan")), 4)),
        ("covid_ece", round(covid.get("ece", float("nan")), 4)),
        ("mrale_rmse", round(mrale.get("rmse", float("nan")), 3)),
        ("mrale_qwk", round(mrale.get("qwk", float("nan")), 4)),
        ("mrale_spearman", round(mrale.get("spearman_rho", float("nan")), 4)),
        ("mrale_within1", round(mrale.get("within1_accuracy", float("nan")), 4)),
        ("mrale_band_accuracy", round(mrale.get("band_accuracy", float("nan")), 4)),
        ("mae_band_none", round(mrale.get("mae_band_none", float("nan")), 3)),
        ("mae_band_mild", round(mrale.get("mae_band_mild", float("nan")), 3)),
        ("mae_band_moderate", round(mrale.get("mae_band_moderate", float("nan")), 3)),
        ("mae_band_severe", round(mrale.get("mae_band_severe", float("nan")), 3)),
        ("median_train_seconds_per_fold", round(float(np.median([
            fold_diagnostics[f"{model_name}/fold{k}"]["train_seconds"]
            for k in range(N_FOLDS) if f"{model_name}/fold{k}" in fold_diagnostics])), 1)),
    ]))

summary = pd.DataFrame(summary_rows)
summary.to_csv(NB06_DIR / "arm_summary.csv", index=False)
cm.write_json(NB06_DIR / "per_fold_metrics.json", dict(per_fold_metrics))

pd.set_option("display.width", 220)
print(summary[["arm", "init", "total_parameters_M", "mrale_mae_pooled", "mrale_mae_ci95",
               "covid_auroc_pooled", "covid_auroc_ci95", "covid_specificity",
               "mrale_qwk"]].to_string(index=False))

## 8. External cohorts

In [ ]:
external_predictions = []
if RUN_EXTERNAL and len(external_cohort):
    external_loader = make_loader(external_cohort, training=False)
    external_records = external_cohort.reset_index(drop=True).to_dict("records")
    for model_name in RUN_MODELS:
        accumulated_score, accumulated_total = None, None
        accumulated_expected, n_models = None, 0
        for fold in range(N_FOLDS):
            checkpoint_path = CHECKPOINT_DIR / f"{model_name}_fold{fold}.pt"
            if not checkpoint_path.is_file():
                continue
            payload = torch.load(checkpoint_path, map_location="cpu")
            backbone, feature_dim, _ = build_backbone(model_name)
            model = MultiTaskCxrModel(backbone, feature_dim)
            model.load_state_dict(payload["state_dict"])
            model = model.to(device)
            collected = run_inference(model, external_loader, external_cohort, device)
            order = collected["index"].tolist()
            reorder = np.argsort(order)
            score = collected["covid_score"][reorder]
            total = collected["mrale_total"][reorder].float()
            expected = collected["mrale_total_expected"][reorder]
            if accumulated_score is None:
                accumulated_score, accumulated_total = score.clone(), total.clone()
                accumulated_expected = expected.clone()
            else:
                accumulated_score += score
                accumulated_total += total
                accumulated_expected += expected
            n_models += 1
            del model, backbone
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        if not n_models:
            continue
        accumulated_score /= n_models
        accumulated_total /= n_models
        accumulated_expected /= n_models
        for position in range(len(external_cohort)):
            source = external_records[position]
            score = float(accumulated_score[position])
            external_predictions.append(cm.make_prediction_row(
                image_key=str(source["image_key"]), cohort=source.get("cohort"),
                subcohort=source.get("subcohort"), filename=source["filename"],
                held_out_fold=None, agent=f"E0g_{model_name}", arm=f"E0g_{model_name}",
                view="v0", task="joint",
                covid_pred="Yes" if score >= 0.5 else "No", covid_score=score,
                mrale_total=int(round(float(accumulated_total[position]))),
                mrale_total_expected=float(accumulated_expected[position]),
                valid=True, parse_error=None,
                model_id=model_name, model_revision=None, ensemble_of_folds=n_models,
                **ground_truth_fields(source),
            ))
        print(f"  {model_name}: {len(external_cohort)} external predictions "
              f"(ensemble of {n_models})")

    cm.write_jsonl(NB06_DIR / "external_predictions.jsonl", external_predictions)
    external_summary = []
    for arm in sorted({row["arm"] for row in external_predictions}):
        for subcohort in sorted({row["subcohort"] for row in external_predictions
                                 if row["arm"] == arm}):
            rows = [row for row in external_predictions
                    if row["arm"] == arm and row["subcohort"] == subcohort]
            covid = evaluate_arm(rows, arm).get("covid", {})
            external_summary.append({
                "arm": arm, "subcohort": subcohort, "n": len(rows),
                "specificity": round(covid.get("specificity", float("nan")), 4),
                "mean_predicted_mrale": round(
                    float(np.mean([row["mrale_total"] for row in rows])), 2),
            })
    if external_summary:
        frame = pd.DataFrame(external_summary)
        frame.to_csv(NB06_DIR / "external_summary.csv", index=False)
        print()
        print(frame.to_string(index=False))
else:
    print("External prediction skipped.")

## 9. Run configuration and gate

In [ ]:
cm.write_json(NB06_DIR / "run_config.json", {
    "written_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "06_conventional_cxr_classifiers.ipynb",
    "protocol_experiments": ["E0g"],
    "seed": SEED,
    "models": {name: MODEL_SPECS[name] for name in RUN_MODELS},
    "model_provenance": model_provenance,
    "training": {
        "epochs": EPOCHS, "batch_size": BATCH_SIZE, "image_size": IMAGE_SIZE,
        "head_lr": LEARNING_RATE, "backbone_lr": BACKBONE_LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY, "warmup_epochs": WARMUP_EPOCHS,
        "patience": PATIENCE, "grad_clip": GRAD_CLIP_NORM, "amp_bf16": USE_AMP,
        "scheduler": "OneCycleLR cosine",
        "selection_criterion": "inner-validation mRALE MAE + 8*(1 - AUROC)",
    },
    "augmentation": {
        "policy": AUGMENTATION,
        "train_manifest": TRAIN_AUGMENTATION_MANIFEST,
        "eval_manifest": EVAL_AUGMENTATION_MANIFEST,
        "forbidden": FORBIDDEN_AUGMENTATIONS,
        "forbidden_rationale": (
            "Horizontal flip is forbidden because extent_right/extent_left are separate "
            "supervised targets: a mirrored image with unmirrored labels teaches the model "
            "that laterality is noise. Mixup/cutmix are forbidden on ordinal mRALE targets."
        ),
    },
    "imbalance_hooks": {
        "class_balanced_sampler": USE_CLASS_BALANCED_SAMPLER,
        "positive_weight": USE_POSITIVE_WEIGHT,
    },
    "fold_diagnostics": fold_diagnostics,
    "checkpoints": checkpoint_paths,
    "smoke_test": SMOKE_TEST,
})

# Usability flags, same convention as NB 07-11 so NB 13/16 read one format.
usability = {}

failures, warnings = [], []

# Forbidden augmentation must be provably inactive.
for name, enabled in FORBIDDEN_AUGMENTATIONS.items():
    if enabled:
        failures.append(f"Forbidden augmentation '{name}' is enabled (protocol Section 5).")
if TRAIN_AUGMENTATION_MANIFEST.get("saturation_hue_jitter", 0.0) != 0.0:
    failures.append("Colour jitter beyond grayscale intensity is active.")

expected_keys = {str(key) for key in internal_cohort["image_key"]}
for model_name in RUN_MODELS:
    model_rows = [row for row in predictions if row["arm"] == f"E0g_{model_name}"]
    covered = {row["image_key"] for row in model_rows}
    if covered != expected_keys:
        failures.append(f"E0g_{model_name}: out-of-fold coverage {len(covered)} of "
                        f"{len(expected_keys)}. Each image must be predicted exactly once.")
    repeated = [key for key, count in Counter(
        row["image_key"] for row in model_rows).items() if count > 1]
    if repeated:
        failures.append(f"E0g_{model_name}: {len(repeated)} duplicate predictions.")
    for fold, metrics in per_fold_metrics[model_name].items():
        auroc = metrics.get("covid", {}).get("auroc")
        if auroc is None or (isinstance(auroc, float) and math.isnan(auroc)):
            failures.append(f"E0g_{model_name} fold {fold}: AUROC not computable.")

densenet_provenance = model_provenance.get("densenet121_xrv", {})
if "densenet121_xrv" in RUN_MODELS and not densenet_provenance.get("domain_pretrained"):
    warnings.append(
        "densenet121_xrv fell back to ImageNet initialisation "
        f"({densenet_provenance.get('torchxrayvision_error')}). Install torchxrayvision "
        "(pip install torchxrayvision) and re-run: an ImageNet CNN is not the "
        "domain-pretrained CXR baseline referee 2d asked for, and Table 2 must not label it "
        "as one."
    )

for model_name in RUN_MODELS:
    for fold in range(N_FOLDS):
        diagnostics = fold_diagnostics.get(f"{model_name}/fold{fold}", {})
        if diagnostics.get("best_epoch") == diagnostics.get("epochs_run", 0) - 1 \
                and diagnostics.get("epochs_run", 0) >= EPOCHS:
            warnings.append(f"{model_name} fold {fold}: best epoch is the last epoch, so "
                            "training was still improving. Consider raising EPOCHS.")

if SMOKE_TEST:
    warnings.append("SMOKE TEST MODE: results are not usable. Set SMOKE_TEST=False.")

# ---- usability flags + Table 2 stamping --------------------------------------------------
for model_name in RUN_MODELS:
    arm = f"E0g_{model_name}"
    arm_rows = [row for row in predictions if row["arm"] == arm]
    if not arm_rows:
        continue
    metrics = evaluate_arm(arm_rows, arm)
    coverage = metrics.get("mrale", {}).get("coverage", float("nan"))
    auroc = metrics.get("covid", {}).get("auroc", float("nan"))
    provenance = model_provenance.get(model_name, {})
    usability[arm] = {
        "score_usable": bool(not math.isnan(auroc)),
        "mrale_usable": bool(not math.isnan(coverage) and coverage > 0.0),
        "mrale_coverage": None if math.isnan(coverage) else round(coverage, 4),
        "domain_pretrained": provenance.get("domain_pretrained"),
        "actual_init": provenance.get("actual_init"),
    }
cm.write_json(NB06_DIR / "usability.json", usability)

summary_path = NB06_DIR / "arm_summary.csv"
if summary_path.is_file() and usability:
    stamped = pd.read_csv(summary_path)
    stamped["score_usable"] = stamped["arm"].map(
        lambda a: usability.get(a, {}).get("score_usable"))
    stamped["mrale_usable"] = stamped["arm"].map(
        lambda a: usability.get(a, {}).get("mrale_usable"))
    stamped["report_in_table2"] = stamped["arm"].map(
        lambda a: "full row" if (usability.get(a, {}).get("score_usable")
                                 and usability.get(a, {}).get("mrale_usable"))
        else "check usability.json")
    # A backbone that silently fell back to ImageNet must not be labelled domain-pretrained.
    stamped["baseline_label"] = stamped["arm"].map(
        lambda a: ("domain-pretrained (TorchXRayVision)"
                   if usability.get(a, {}).get("domain_pretrained")
                   else "ImageNet-initialised"))
    stamped.to_csv(summary_path, index=False)
    print()
    print("arm_summary.csv stamped:")
    print(stamped[["arm", "baseline_label", "mrale_usable", "report_in_table2"]]
          .to_string(index=False))


def report(title, messages):
    print(title)
    if messages:
        for message in messages:
            print("  -", message)
    else:
        print("  none")


report("WARNINGS", warnings)
print()
report("FAILURES", failures)

cm.write_json(NB06_DIR / "gate_nb06.json",
              {"passed": not failures, "failures": failures, "warnings": warnings})
# Reasons go in the message so a pasted traceback is self-explanatory.
if failures:
    detail = "\n".join(f"  [{index + 1}] {message}"
                       for index, message in enumerate(failures))
    raise AssertionError(
        f"NB 06 gate failed with {len(failures)} blocking issue(s):\n{detail}")
print()
print("NB 06 gate: PASSED")

## Notes carried forward

- These are **baseline** rows in Table 2, not framework agents. They are deliberately excluded
  from the E1 agent roster: adding a supervised CNN as an agent would confound "does the
  multi-agent framework help" with "does adding any extra model help".
- `torchxrayvision` availability is the one thing to check before quoting DenseNet-121. The
  gate warns rather than blocks, but a fallback to ImageNet must be disclosed or the baseline
  is mislabelled.
- Per-model parameter counts, peak memory, and median per-fold training seconds are in
  `run_config.json` for Table 10. The comparison worth making explicit: these models train in
  minutes and run in milliseconds, against a 4B VLM that needs seconds per image. If a
  conventional baseline is within the confidence interval of the framework, that operational
  gap belongs in the discussion.
- `arm_summary.csv` has the same column names as NB 05's, so NB 16 can concatenate the two
  without renaming anything.